# HOGformer + soft-voting HOG (proposed) on Kaggle - training from scratch
Session options (right sidebar): **Accelerator = GPU T4 x2**, **Internet = On**, **Persistence = Files only**, Environment = latest.
No checkpoint dataset is needed (training starts from iteration 0).
Run cells 1-6, or use **Save Version -> Save & Run All (Commit)** to run in the background (~3.5-4 h).

In [ ]:
# 1) GPU and disk check
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!df -h /tmp /kaggle/working

In [ ]:
# 2) Code + dependencies
%cd /kaggle/working
!test -d HOGformer_DLCV || git clone -b claude/lucid-fermi-emhnbo https://github.com/adityaj1706/HOGformer_DLCV.git
%cd /kaggle/working/HOGformer_DLCV
!git pull
!git log -1 --oneline
!pip install -q -r settingI/requirements_colab.txt
ALLWEATHER_ID = '1tfeBnjZX1wIhIFPl6HOzzOKOyo0GdGHl'   # replace with the id of YOUR shared copy if this hits the Drive quota

In [ ]:
# 3) Data -> /tmp/data (not /kaggle/working, which is capped and saved as output).
#    /content/data is a symlink so the configs are IDENTICAL to the Colab runs.
!mkdir -p /tmp/data /content
!ln -sfn /tmp/data /content/data
%cd /kaggle/working/HOGformer_DLCV
!SKIP_SNOW=1 ALLWEATHER_ID={ALLWEATHER_ID} bash scripts/download_data.sh /tmp/data

In [ ]:
# 5) START training in the BACKGROUND (this cell returns immediately).
#    Change CONFIG to ..._A.yml / ..._B.yml for the proposed models.
import subprocess, os
CONFIG = 'Allweather/Options/Allweather_HOGformer_softvote.yml'
os.chdir('/kaggle/working/HOGformer_DLCV/settingI')
!ln -sfn /kaggle/working/experiments experiments
proc = subprocess.Popen(f'python -u basicsr/train.py -opt {CONFIG} --launcher none > /kaggle/working/train_stdout.log 2>&1', shell=True)
print('started, pid', proc.pid)

In [ ]:
# 5b) MONITOR - run this cell any time (it does not interrupt training)
!tail -n 40 /kaggle/working/train_stdout.log | grep -E "iter: +[0-9]+0,|Validation|Resuming|End of training|Error|Traceback" | tail -4 | cut -c1-210
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv
print('still running' if proc.poll() is None else f'FINISHED, exit code {proc.returncode}')

In [ ]:
# 5c) WAIT until training finishes (needed for "Save & Run All"; optional when interactive)
import time
while proc.poll() is None:
    time.sleep(30)
print('training finished, exit code', proc.returncode)

In [ ]:
# 6) What was saved
!ls -la /kaggle/working/experiments/Allweather_HOGformer_softvote/models | tail -4
!grep -h Validation /kaggle/working/experiments/Allweather_HOGformer_softvote/train_*.log | tail -6